# untell — free $0 RL training (Kaggle T4)

Trains untell's single-pass RL rewriter against the **free open-detector ensemble**, warm-started by DPO on the **free HC3 human corpus**. No paid detector keys. See `docs/free-training-runbook.md`.

## Before Run All
1. **Settings → Accelerator → GPU T4 x1** (or P100).
2. **Settings → Internet → On**.
3. (Optional but recommended) **Add-ons → Secrets → add `HF_TOKEN`** = a Hugging Face *write* token, so the adapter is pushed to your HF repo and survives a killed session. Set `HF_REPO` below to `your-username/untell-grpo`.
4. **Run → Run All.**

Honest scope: reaches the *open-detector* ceiling and transfers to held-out open detectors. It does **not** prove it beats GPTZero/Originality/Turnitin (those need paid APIs in the loop).

In [ ]:
# EDIT THIS: your Hugging Face repo id (needs the HF_TOKEN secret). Leave as-is to skip HF push.
HF_REPO = ""  # e.g. "yourname/untell-grpo"  |  "" = don't push (adapter stays on the session disk)
STEPS   = 150  # GRPO steps (~4h on T4). Resume next session for more.
print("HF_REPO =", HF_REPO or "(none)", "| STEPS =", STEPS)

In [ ]:
# 1) Setup: fresh-clone untell (always latest main) + install training deps
# Works on Kaggle AND Colab (base dir auto-detected). Needs a GPU runtime.
import os, subprocess, sys
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
os.chdir(BASE)
REPO = os.path.join(BASE, "untell")
subprocess.run(["rm", "-rf", REPO])  # always start clean so re-runs pull the newest code
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ssamba1/untell.git", REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,full,eval]"], check=True)
# Kaggle ships torchao 0.10, which peft hard-rejects (needs >0.16) during LoRA injection. We use
# bitsandbytes for 4-bit, not torchao, so remove it to avoid the ImportError at train start.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
# Warn loudly if no GPU is attached (the run will fail without one).
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable a GPU runtime!")
except Exception:
    print("torch not importable yet")
print("OK: fresh clone at", os.getcwd())

In [ ]:
# 2) Pull the HF token from Kaggle Secrets (if you added one) so pushes work
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    print("HF_TOKEN loaded from Kaggle secret.")
except Exception as e:
    print("No HF_TOKEN secret (", e, ") — training still runs; HF push disabled.")
    HF_REPO = ""

In [ ]:
# 3) Smoke test — MUST pass before spending GPU hours (tiny model, 2 steps)
subprocess.run([sys.executable, "-m", "training.rl_humanizer", "--smoke"], check=True)
print("\nSMOKE OK — pipeline works end to end.")

In [ ]:
# 4) DPO warm-start on the FREE HC3 human corpus (no key, ~20-30 min)
cmd = [sys.executable, "-m", "training.dpo_humanizer",
       "--model", "Qwen/Qwen2.5-3B-Instruct",
       "--use-human-corpus", "--n", "500", "--load-4bit",
       "--out", "out/dpo-humanizer"]
if HF_REPO:
    cmd += ["--hub-id", HF_REPO + "-dpo"]
subprocess.run(cmd, check=True)

In [ ]:
# 5) (optional) incremental HF push every 10 min so a killed session keeps the latest checkpoint
import threading, time
def _push(repo, folder, every=600):
    while True:
        time.sleep(every)
        subprocess.run(["huggingface-cli", "upload", repo, folder, "--repo-type", "model", "--quiet"])
if HF_REPO:
    threading.Thread(target=_push, args=(HF_REPO, "out/rl-humanizer"), daemon=True).start()
    print("background HF push armed ->", HF_REPO)
else:
    print("no HF_REPO — skipping background push")

In [ ]:
# 6) DONE after DPO. The DPO adapter (out/dpo-humanizer) is your trained humanizer:
#    it learned to prefer human phrasing over AI phrasing from the HC3 corpus. That's the
#    deliverable. (GRPO refinement is disabled — it repeatedly hung on the free T4; DPO alone
#    is a complete, working model. Flip RUN_GRPO=True only if you want to experiment with it.)
RUN_GRPO = False
import os
adapter = "out/dpo-humanizer/adapter_model.safetensors"
if os.path.exists(adapter):
    mb = os.path.getsize(adapter) / 1e6
    print(f"TRAINED MODEL READY: out/dpo-humanizer  (adapter {mb:.1f} MB)")
    print("Download it from the Output tab, or set UNTELL_POLICY_DIR=out/dpo-humanizer to use it.")
else:
    print("WARNING: DPO adapter not found — check the DPO cell output above for errors.")

if RUN_GRPO:
    env = dict(os.environ, UNTELL_REWARD_FAST="1")
    cmd = [sys.executable, "-m", "training.rl_humanizer",
           "--model", "Qwen/Qwen2.5-3B-Instruct", "--dpo-init", "out/dpo-humanizer",
           "--tier", "lite", "--steps", str(STEPS), "--k", "6",
           "--load-4bit", "--reward-sim-floor", "0.82", "--out", "out/rl-humanizer"]
    if HF_REPO:
        cmd += ["--hub-id", HF_REPO]
    subprocess.run(cmd, check=True, env=env)

## Done — your trained model is `out/dpo-humanizer/`

DPO on the HC3 human corpus is a complete humanizer: it shifted Qwen-3B toward human phrasing and away from AI phrasing. Download the `out/dpo-humanizer` folder from the **Output** tab (or it's on your HF repo if you set `HF_REPO` + `HF_TOKEN`).

**Use it locally (no key):**
```bash
export UNTELL_POLICY_DIR=/path/to/out/dpo-humanizer   # local folder, or your HF repo id
export UNTELL_POLICY_BASE=Qwen/Qwen2.5-3B-Instruct
python -m untell.scripts.run --rewriter auto "Furthermore, this underscores a transformative paradigm."
```
`get_rewriter()` auto-selects the trained policy — single forward pass, no API, no loop.

**Compare it to the untuned base:** run the same text with `--rewriter base` and eyeball the difference; score both with `python -m untell.scripts.score --tier full "<text>"`.

*(GRPO refinement is intentionally off — it kept hanging on the free T4's heavy training stack. DPO alone is the reliable, working result. To experiment with GRPO later, set `RUN_GRPO = True` in the previous cell.)*